In [1]:
from transformers import CLIPModel, CLIPProcessor
import torch
from PIL import Image
import os
import torch.nn.functional as F
import numpy as np
import pandas as pd

In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM

## IMPORT & EXPLORE

#### GEMMA SETUP

In [3]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

gemma_1b = Gemma3ForCausalLM.from_pretrained(
    "google/gemma-3-1b-it", quantization_config=quantization_config
).eval()

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

#### CLIP SETUP

In [4]:
# Load pretrained CLIP
clip_model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_id)
clip_processor = CLIPProcessor.from_pretrained(clip_model_id, use_fast = True)

In [5]:
clip_model = clip_model.to('cuda')

In [6]:
clip_model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

#### IMAGE DIR SETUP

In [36]:
class ImageFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        # recursively collect all image files in all subfolders
        self.image_files = []
        for root, _, files in os.walk(root_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    self.image_files.append(os.path.join(root, f))
        
    
    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        
        image_path = self.image_files[index]
        image = Image.open(image_path).convert('RGB')
        
        clip_processed_image = clip_processor( images = image, return_tensors = 'pt')

        return clip_processed_image, image_path


In [37]:
image_dataset = ImageFolderDataset(r"C:\Users\User\Downloads\data\images")

In [38]:
len(image_dataset)

5782

In [45]:
image_dataset[42][0]

{'pixel_values': tensor([[[[1.6092, 1.6092, 1.6092,  ..., 1.5946, 1.5946, 1.5946],
          [1.6092, 1.6092, 1.6092,  ..., 1.5946, 1.5946, 1.5946],
          [1.5946, 1.5946, 1.5946,  ..., 1.5946, 1.5946, 1.5946],
          ...,
          [1.5800, 1.5800, 1.5800,  ..., 1.6092, 1.6092, 1.6092],
          [1.5800, 1.5800, 1.5800,  ..., 1.6092, 1.6092, 1.6092],
          [1.5800, 1.5800, 1.5800,  ..., 1.6092, 1.6092, 1.6092]],

         [[1.7747, 1.7747, 1.7747,  ..., 1.7597, 1.7597, 1.7597],
          [1.7747, 1.7747, 1.7747,  ..., 1.7597, 1.7597, 1.7597],
          [1.7597, 1.7597, 1.7597,  ..., 1.7597, 1.7597, 1.7597],
          ...,
          [1.7747, 1.7747, 1.7747,  ..., 1.7747, 1.7747, 1.7747],
          [1.7747, 1.7747, 1.7747,  ..., 1.7747, 1.7747, 1.7747],
          [1.7747, 1.7747, 1.7747,  ..., 1.7747, 1.7747, 1.7747]],

         [[1.8473, 1.8473, 1.8473,  ..., 1.8331, 1.8331, 1.8331],
          [1.8473, 1.8473, 1.8473,  ..., 1.8331, 1.8331, 1.8331],
          [1.8331, 1.8331

In [51]:
images_dataloader = torch.utils.data.DataLoader( dataset = image_dataset, 
                                                 batch_size = 64 )

## COMPUTING EMBEDDINGS

In [54]:
image_embeddings_list = []
image_paths_list = []

clip_model.eval()

with torch.no_grad():
    for batch_num, (batch_images, batch_image_paths) in enumerate(images_dataloader):
        # Move to GPU
        batch_images = {k: v.to("cuda") for k, v in batch_images.items()}
        
        # Fix extra dimension if present
        pixel_values = batch_images["pixel_values"]
        if pixel_values.ndim == 5:   # [batch, 1, 3, H, W]
            pixel_values = pixel_values.squeeze(1)  # -> [batch, 3, H, W]
            # Update the batch_images dictionary with the corrected tensor
            batch_images["pixel_values"] = pixel_values
        
        # Get embeddings from CLIP
        image_embeddings = clip_model.get_image_features(**batch_images)
        
        # Normalize (important for cosine sim later)
        image_embeddings = image_embeddings / image_embeddings.norm(p=2, dim=-1, keepdim=True)
        
        # Store Paths and Image embeddings 
        image_embeddings_list.append(image_embeddings)
        image_paths_list.extend(batch_image_paths)
        
        print(f'Processed {batch_num + 1} / {len(images_dataloader)}')

Processed 1 / 91
Processed 2 / 91
Processed 3 / 91
Processed 4 / 91
Processed 5 / 91
Processed 6 / 91
Processed 7 / 91
Processed 8 / 91
Processed 9 / 91
Processed 10 / 91
Processed 11 / 91
Processed 12 / 91
Processed 13 / 91
Processed 14 / 91
Processed 15 / 91
Processed 16 / 91
Processed 17 / 91
Processed 18 / 91
Processed 19 / 91
Processed 20 / 91
Processed 21 / 91
Processed 22 / 91
Processed 23 / 91
Processed 24 / 91
Processed 25 / 91
Processed 26 / 91
Processed 27 / 91
Processed 28 / 91
Processed 29 / 91
Processed 30 / 91
Processed 31 / 91
Processed 32 / 91
Processed 33 / 91
Processed 34 / 91
Processed 35 / 91
Processed 36 / 91
Processed 37 / 91
Processed 38 / 91
Processed 39 / 91
Processed 40 / 91
Processed 41 / 91
Processed 42 / 91
Processed 43 / 91
Processed 44 / 91
Processed 45 / 91
Processed 46 / 91
Processed 47 / 91
Processed 48 / 91
Processed 49 / 91
Processed 50 / 91
Processed 51 / 91
Processed 52 / 91
Processed 53 / 91
Processed 54 / 91
Processed 55 / 91
Processed 56 / 91
P

In [59]:
image_embeddings_list

[tensor([[-0.0290, -0.0018,  0.0210,  ...,  0.0775, -0.0089,  0.0477],
         [-0.0326,  0.0380,  0.0062,  ...,  0.0049, -0.0174,  0.0510],
         [-0.0234,  0.0099,  0.0263,  ...,  0.0036,  0.0020,  0.0441],
         ...,
         [-0.0401, -0.0105,  0.0180,  ...,  0.0113,  0.0133,  0.0331],
         [ 0.0025, -0.0297,  0.0138,  ..., -0.0080,  0.0119,  0.0462],
         [ 0.0028,  0.0177,  0.0279,  ...,  0.0323,  0.0070,  0.0437]],
        device='cuda:0'),
 tensor([[ 0.0008,  0.0126,  0.0250,  ...,  0.0338,  0.0002,  0.0364],
         [-0.0234, -0.0131,  0.0232,  ...,  0.0152,  0.0068,  0.0476],
         [-0.0170, -0.0203,  0.0077,  ..., -0.0070,  0.0222,  0.0447],
         ...,
         [-0.0408,  0.0665,  0.0290,  ...,  0.0468,  0.0196,  0.0151],
         [-0.0256,  0.0390,  0.0155,  ...,  0.0582, -0.0188,  0.0085],
         [-0.0425,  0.0362,  0.0352,  ...,  0.0519, -0.0022,  0.0231]],
        device='cuda:0'),
 tensor([[-0.0305,  0.0257,  0.0236,  ...,  0.0792, -0.0024,  0.01

In [60]:
all_image_embeddings = torch.cat(image_embeddings_list, dim=0)

In [62]:
all_image_embeddings.shape

torch.Size([5782, 512])

In [63]:
torch.save( {'image_embeddings' : all_image_embeddings.cpu(),
             'image_paths' : image_paths_list},
             
             r'C:\Users\User\Downloads\clip_image_embeddings.pt'  )

## SEARCH REQUEST

#### LOAD DATA

In [7]:
clip_image_data = torch.load(r"C:\Users\User\Downloads\clip_image_embeddings.pt")

In [8]:
clip_image_embeddings = clip_image_data['image_embeddings']
image_paths = clip_image_data['image_paths']

In [9]:
clip_image_embeddings.shape[0] == len(image_paths)

True

#### SEND REQUEST TO GEMMA

In [24]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a fashion assistant for men's clothing only MUST respond with ONLY a valid Python list of exactly 4 clothing items.\n\n"
            
            "RESPONSE FORMAT - You must respond with EXACTLY this format (no other text):\n"
            '["bottom item", "top item", "outer layer item", "shoe item"]\n\n'
            
            "EXAMPLES OF CORRECT RESPONSES:\n"
            '["black jeans", "white t-shirt", "grey hoodie", "white sneakers"]\n'
            '["blue chinos", "navy polo shirt", "black jacket", "brown loafers"]\n'
            '["dark joggers", "black t-shirt", "black zip-hoodie", "black boots"]\n\n'
            
            "CATEGORIES (pick exactly ONE from each):\n"
            "1. BOTTOMS: jeans, chinos, joggers, shorts, trousers\n"
            "2. TOPS: t-shirt, polo shirt, tank top\n"
            "3. OUTER LAYERS: hoodie, zip-hoodie, sweatshirt, cardigan, jacket\n"
            "4. SHOES: sneakers, boots, loafers, dress shoes, canvas shoes\n\n"
            
            "RULES:\n"
            "- Return ONLY the Python list, nothing else\n"
            "- Use simple item names with colors (e.g. 'black jeans', 'white sneakers')\n"
            "- If user specifies color theme (like 'total black'), ALL items must match\n"
            "- Always include color in the item name\n"
            "- No quotes around the entire response, just the list\n"
            "- No explanations, no extra text, no formatting"
        )
    },
    {
        "role": "user", 
        "content": "light summer"
    }
]

In [25]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to('cuda')

with torch.inference_mode():
    outputs = gemma_1b.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)

In [26]:
outputs

['<bos><start_of_turn>user\nYou are a fashion assistant for men\'s clothing only MUST respond with ONLY a valid Python list of exactly 4 clothing items.\n\nRESPONSE FORMAT - You must respond with EXACTLY this format (no other text):\n["bottom item", "top item", "outer layer item", "shoe item"]\n\nEXAMPLES OF CORRECT RESPONSES:\n["black jeans", "white t-shirt", "grey hoodie", "white sneakers"]\n["blue chinos", "navy polo shirt", "black jacket", "brown loafers"]\n["dark joggers", "black t-shirt", "black zip-hoodie", "black boots"]\n\nCATEGORIES (pick exactly ONE from each):\n1. BOTTOMS: jeans, chinos, joggers, shorts, trousers\n2. TOPS: t-shirt, polo shirt, tank top\n3. OUTER LAYERS: hoodie, zip-hoodie, sweatshirt, cardigan, jacket\n4. SHOES: sneakers, boots, loafers, dress shoes, canvas shoes\n\nRULES:\n- Return ONLY the Python list, nothing else\n- Use simple item names with colors (e.g. \'black jeans\', \'white sneakers\')\n- If user specifies color theme (like \'total black\'), ALL i

In [17]:
sum(p.numel() for p in gemma_1b.parameters())

999885952